# Problema 2: Análisis de Vulnerabilidades con Web Scraping

**Objetivo:** scrapear los avisos de seguridad de Debian (Debian Security Advisories) de los últimos 3 meses desde `https://lists.debian.org/debian-security-announce/`, extraer CVE ID, fecha de publicación, paquete afectado y descripción; clasificar el tipo de vulnerabilidad de cada aviso a partir de su descripción; y dejar los datos listos para conectar a Looker Studio.

**Estructura del notebook:**
1. Scraping de los avisos (extracción de HTML → dataset crudo).
2. Clasificación del tipo de vulnerabilidad (embeddings semánticos + reglas de respaldo).
3. Preparación del dataset final para Looker Studio.

## 1. Dependencias e imports

In [1]:
!pip install sentence-transformers pandas scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


`requests` + `BeautifulSoup` para el scraping, `re` para parsear el texto de cada aviso, y `dateutil.relativedelta` para calcular la ventana de "últimos 3 meses" en base a la fecha actual.

In [2]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, date
from dateutil.relativedelta import relativedelta

## 2. Scraping de los avisos de seguridad

### Estructura de la fuente

`https://lists.debian.org/debian-security-announce/<año>/` es un índice HTML con un link (`msgNNNNN.html`) por cada aviso publicado ese año. Cada aviso individual es una página de texto con formato semi-estructurado (`Package:`, `CVE ID:`, fecha, descripción libre, secciones de "For the stable distribution..."), no HTML estructurado — por eso la extracción se hace con expresiones regulares sobre el texto plano de la página, en vez de parsear tablas.

### Ventana de tiempo y el índice por año

El enunciado pide los avisos de los **últimos 3 meses**. El índice de avisos está partido **por año**, así que si la ventana de 3 meses cruza un cambio de año (por ejemplo, corriendo el scraper en enero, donde "los últimos 3 meses" incluye noviembre y diciembre del año anterior), hace falta scrapear el índice de dos años distintos, no solo el del año actual. Lo resolvemos calculando qué años cubre la ventana `[FECHA_INICIO, FECHA_FIN]` y armando dinámicamente la URL del índice de cada uno, en vez de hardcodear un único año.

### Función de extracción por aviso

Por cada aviso extraemos:
- **Fecha**: la buscamos en las primeras líneas del texto, con el formato `"Month DD, YYYY"` que usa Debian.
- **Package**: línea `Package: <nombre>`.
- **CVE(s)**: un aviso puede listar **más de un CVE** en el mismo bloque `CVE ID:` (uno por línea) — por eso extraemos todos los que hagan match con el patrón `CVE-AAAA-NNNN` dentro de ese bloque, no solo el primero.
- **Descripción**: el texto libre entre el bloque de metadatos y la sección estándar "For the stable distribution..." (o alguna de sus variantes, según la distribución de Debian que corresponda). Si ese patrón no aparece (avisos con formato menos común), usamos como respaldo una extracción más simple basada en el campo `Debian Bug`.

In [3]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

BASE_URL_TEMPLATE = "https://lists.debian.org/debian-security-announce/{year}/"

FECHA_FIN = date.today()
FECHA_INICIO = FECHA_FIN - relativedelta(months=3)

# Años que cubre la ventana de 3 meses (normalmente uno solo; dos si la ventana
# cruza un cambio de año, ej. corriendo el scraper en enero/febrero/marzo).
ANIOS_A_SCRAPEAR = sorted({FECHA_INICIO.year, FECHA_FIN.year}, reverse=True)

print("Fecha inicio:", FECHA_INICIO)
print("Fecha fin:", FECHA_FIN)
print("Años a scrapear:", ANIOS_A_SCRAPEAR)


# ============================================================
# FUNCIÓN PARA EXTRAER LOS DATOS DE UN AVISO
# ============================================================

def extraer_datos(texto):

    # ---- Fecha ----
    fecha_match = re.search(
        r"Debian Security Advisory DSA-\d+-\d+.*?\n.*?\n.*?\n([A-Z][a-z]+ \d{1,2}, \d{4})",
        texto,
        re.DOTALL
    )
    fecha = None
    if fecha_match:
        fecha = datetime.strptime(fecha_match.group(1), "%B %d, %Y").date()

    # ---- Package ----
    package_match = re.search(r"Package\s*:\s*(.+)", texto)
    package = package_match.group(1).strip() if package_match else None

    # ---- CVE(s) ----
    # El bloque "CVE ID:" puede listar varios CVE (uno por línea), así que
    # extraemos TODOS los que matcheen el patrón dentro de ese bloque.
    cve_match = re.search(
        r"CVE ID\s*:\s*([\s\S]+?)(?=\n[A-Z][a-z]+|\r?\n-{3,}|$)",
        texto
    )
    if cve_match:
        cves_encontrados = re.findall(r"CVE-\d{4}-\d+", cve_match.group(1))
        cves_unicos = list(dict.fromkeys(cves_encontrados))  # sin duplicados, preserva orden
        cves = " ".join(cves_unicos) if cves_unicos else "not yet available"
    else:
        cves = "not yet available"

    # ---- Descripción ----
    descripcion = None
    inicio_match = re.search(r"CVE ID\s*:\s*.*?\n", texto)

    if inicio_match:
        texto_posterior = texto[inicio_match.end():]

        # Sacamos campos de metadatos adicionales que puedan venir después del CVE ID
        texto_posterior = re.sub(
            r"^\s*Debian Bug\s*:.*?\n",
            "",
            texto_posterior,
            flags=re.MULTILINE
        ).strip()

        # La descripción termina donde empieza alguna de las secciones estándar
        # de recomendaciones de Debian (varía según la distribución del aviso).
        patrones_fin = [
            r"\nFor the stable distribution",
            r"\nFor the oldstable distribution",
            r"\nFor the oldoldstable distribution",
            r"\nWe recommend that you upgrade",
            r"\nFor the detailed security status",
        ]
        posiciones = [
            m.start()
            for patron in patrones_fin
            if (m := re.search(patron, texto_posterior))
        ]
        fin = sorted(posiciones)[0] if posiciones else len(texto_posterior)

        descripcion = " ".join(texto_posterior[:fin].strip().split())
    else:
        # Fallback: algunos avisos no siguen el formato estándar con "CVE ID:";
        # probamos extraer la descripción a partir del campo "Debian Bug".
        descripcion_match = re.search(
            r"Debian Bug\s*:.*?\n\n(.*?)\n\nFor the stable distribution",
            texto,
            re.DOTALL
        )
        if descripcion_match:
            descripcion = " ".join(descripcion_match.group(1).split())

    return {
        "fecha": fecha,
        "package": package,
        "cves": cves,
        "descripcion": descripcion
    }

Fecha inicio: 2026-06-20
Fecha fin: 2026-09-20
Años a scrapear: [2026]


### Obtención de los links de cada índice anual

Por cada año que cubre la ventana de 3 meses, bajamos su página de índice y extraemos todos los links a avisos individuales (`msgNNNNN.html`). Los ordenamos de forma descendente **dentro de cada año** — la numeración de mensajes se reinicia año a año, así que no tendría sentido ordenar de forma global mezclando números de años distintos; en cambio, procesamos primero el año más reciente (y dentro de él, del aviso más nuevo al más viejo), y recién después el año anterior si corresponde.

In [4]:
# ============================================================
# OBTENER LINKS DE CADA ÍNDICE ANUAL
# ============================================================

# Cada elemento es (url_del_indice_de_ese_anio, href_del_aviso), para poder
# reconstruir la URL completa de cada aviso más adelante.
links = []

for anio in ANIOS_A_SCRAPEAR:

    url_indice = BASE_URL_TEMPLATE.format(year=anio)

    response = requests.get(url_indice)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    links_anio = [
        a.get("href")
        for a in soup.find_all("a")
        if a.get("href") and re.match(r"msg\d+\.html$", a.get("href"))
    ]

    # Orden descendente dentro del año (ver nota en la celda de markdown anterior
    # sobre por qué no se puede ordenar de forma global entre años distintos)
    links_anio.sort(
        key=lambda href: int(re.search(r"msg(\d+)\.html", href).group(1)),
        reverse=True
    )

    print(f"Año {anio}: {len(links_anio)} avisos encontrados")

    links.extend((url_indice, href) for href in links_anio)

print("\nCantidad total de avisos encontrados:", len(links))

Año 2026: 421 avisos encontrados

Cantidad total de avisos encontrados: 421


### Recorrido de los avisos y filtro de ventana temporal

Procesamos los avisos del más nuevo al más viejo (año actual primero, y dentro de cada año en orden descendente). Esto permite una optimización: en cuanto encontramos un aviso **anterior** a `FECHA_INICIO`, ya sabemos que todos los que siguen también lo son (porque vamos en orden descendente), así que cortamos el scraping ahí (`break`) en vez de seguir pidiendo páginas de más.

In [5]:
# ============================================================
# RECORRER LOS AVISOS
# ============================================================

resultados = []


for i, (url_indice, href) in enumerate(links):

    url = url_indice + href

    print(f"\n[{i + 1}/{len(links)}] Procesando {href}")

    try:

        response = requests.get(url)

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        texto = soup.get_text("\n", strip=True)

        datos = extraer_datos(texto)


        # Si no pudimos obtener la fecha,
        # continuamos con el siguiente aviso

        if datos["fecha"] is None:

            print("  No se pudo obtener la fecha")
            continue


        print("  Fecha:", datos["fecha"])
        print("  Package:", datos["package"])
        print("  CVE:", datos["cves"])


        # ====================================================
        # FILTRO DE LOS ÚLTIMOS 3 MESES
        # ====================================================

        if datos["fecha"] > FECHA_FIN:

            continue


        if datos["fecha"] < FECHA_INICIO:

            print("  Aviso anterior a los últimos 3 meses.")
            print("  Deteniendo scraping.")

            break


        # ====================================================
        # GUARDAR
        # ====================================================

        datos["dsa"] = re.search(
            r"DSA-\d+-\d+",
            texto
        ).group(0)

        datos["url"] = url

        resultados.append(datos)


    except requests.RequestException as e:

        print("  Error:", e)


[1/421] Procesando msg00420.html
  Fecha: 2026-09-19
  Package: chromium
  CVE: CVE-2026-93372 CVE-2026-93373 CVE-2026-93374 CVE-2026-93375 CVE-2026-93376 CVE-2026-93377 CVE-2026-93378 CVE-2026-93379 CVE-2026-93380 CVE-2026-93381 CVE-2026-93382 CVE-2026-93383 CVE-2026-93384 CVE-2026-93385 CVE-2026-93386 CVE-2026-93387

[2/421] Procesando msg00419.html
  Fecha: 2026-09-19
  Package: unbound
  CVE: CVE-2026-14586 CVE-2026-32665 CVE-2026-40622 CVE-2026-40691 CVE-2026-41637 CVE-2026-42955 CVE-2026-44621 CVE-2026-44687 CVE-2026-44690 CVE-2026-46582 CVE-2026-50045 CVE-2026-50046 CVE-2026-50243 CVE-2026-50248 CVE-2026-50251 CVE-2026-50252 CVE-2026-52863 CVE-2026-54478 CVE-2026-55708 CVE-2026-55717 CVE-2026-55973 CVE-2026-55990 CVE-2026-55991 CVE-2026-56416 CVE-2026-56444 CVE-2026-77860 CVE-2026-77955 CVE-2026-78227 CVE-2026-80225 CVE-2026-81634 CVE-2026-81642 CVE-2026-82717 CVE-2026-82720 CVE-2026-85501

[3/421] Procesando msg00418.html
  Fecha: 2026-09-17
  Package: chromium
  CVE: CVE-2026

### Consolidación y guardado

Armamos el DataFrame final, lo ordenamos por fecha, y lo guardamos como el dataset crudo (`debian_security_advisories_ultimos_3_meses.csv`) — este es el entregable que responde directamente al primer punto del enunciado (CVE ID, fecha, paquete, descripción).

In [6]:
# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(resultados)


# Ordenamos por fecha

df = df.sort_values(
    "fecha",
    ascending=False
).reset_index(drop=True)


print("\n===================================")
print("RESULTADO")
print("===================================")

print("Cantidad de avisos:", len(df))

print(df.head())


# ============================================================
# GUARDAR CSV
# ============================================================

df.to_csv(
    "debian_security_advisories_ultimos_3_meses.csv",
    index=False
)

print(
    "\nArchivo guardado como:",
    "debian_security_advisories_ultimos_3_meses.csv"
)


RESULTADO
Cantidad de avisos: 154
        fecha                      package  \
0  2026-09-19                     chromium   
1  2026-09-19                      unbound   
2  2026-09-17                     chromium   
3  2026-09-17  libapache2-mod-auth-openidc   
4  2026-09-17                        bind9   

                                                cves  \
0  CVE-2026-93372 CVE-2026-93373 CVE-2026-93374 C...   
1  CVE-2026-14586 CVE-2026-32665 CVE-2026-40622 C...   
2  CVE-2026-87429 CVE-2026-87430 CVE-2026-87431 C...   
3                                     CVE-2026-54789   
4  CVE-2026-19033 CVE-2026-19662 CVE-2026-19666 C...   

                                         descripcion         dsa  \
0  CVE-2026-93376 CVE-2026-93377 CVE-2026-93378 C...  DSA-6508-1   
1  CVE-2026-41637 CVE-2026-42955 CVE-2026-44621 C...  DSA-6507-1   
2  CVE-2026-87433 CVE-2026-87434 CVE-2026-87435 C...  DSA-6506-1   
3  It was discovered that incorrect state cookie ...  DSA-6504-1   
4  CVE-2026

## 3. Clasificación del tipo de vulnerabilidad

**Enfoque elegido: similitud semántica por embeddings, con respaldo por palabras clave.**

El enunciado pide clasificar cada aviso en una o más categorías (denial of service, ejecución remota de código, bypass de restricciones, exposición de información, etc.) a partir de su descripción en texto libre. Evaluamos tres formas de hacerlo:

1. **Reglas de palabras clave puras** (buscar substrings como "denial of service" en la descripción): simple y rápido, pero frágil — no captura variaciones de redacción ("service disruption", "crash", "resource exhaustion" para DoS, por ejemplo) y requeriría mantener a mano una lista enorme de sinónimos por categoría.
2. **Un LLM clasificando cada descripción** (similar al Enfoque A del Problema 1): más flexible, pero implica una llamada a API por cada aviso (costo y latencia), y es menos determinístico/reproducible para una tarea de clasificación con categorías fijas y conocidas de antemano.
3. **Similitud semántica por embeddings** (la elegida): se define un catálogo de categorías, cada una con un nombre y sinónimos; se generan los embeddings del catálogo una sola vez, se compara contra el embedding de cada descripción por similitud coseno, y se agrega como respaldo la búsqueda literal de esos mismos sinónimos (por si el embedding no llega al umbral pero el término aparece explícito en el texto). Es determinístico, no requiere llamadas a API por fila, y generaliza mejor que las reglas puras porque compara *significado*, no solo substrings exactos.

**Por qué multi-label:** una misma vulnerabilidad puede pertenecer a más de una categoría a la vez (por ejemplo, un bug que permite tanto denial of service como ejecución remota de código). Por eso la clasificación devuelve una **lista** de categorías por aviso, no una única etiqueta.

**Sobre el umbral (`umbral=0.28`):** el valor se ajustó empíricamente probando contra avisos del propio dataset — es intencionalmente permisivo (en vez de, por ejemplo, 0.5) porque las descripciones de Debian son técnicas y cortas, y un umbral más estricto dejaba afuera clasificaciones correctas. El respaldo por palabras clave (paso 3 de la función de clasificación) compensa los falsos negativos que ese umbral permisivo no llega a cubrir.

Reimportamos las librerías necesarias para esta sección, para que el notebook pueda correrse desde acá si ya se tiene generado el CSV crudo del scraping.

In [7]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Juli\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Catálogo de categorías

Definimos 16 categorías de vulnerabilidad comunes (alineadas con clasificaciones estándar de la industria), cada una con su nombre canónico y una lista de sinónimos/variantes de redacción, que sirven tanto para generar el embedding de referencia de la categoría como para el respaldo por palabras clave.

In [8]:
# 1. Catálogo enriquecido con mapeo semántico para mejorar la coincidencia

categorias_mapeo = {

    "Denial of Service": [
        "Denial of Service",
        "service disruption",
        "crash",
        "infinite loop",
        "resource exhaustion"
    ],

    "Remote Code Execution": [
        "Remote Code Execution",
        "execution of arbitrary code",
        "arbitrary code execution",
        "execute arbitrary code"
    ],

    "Privilege Escalation": [
        "Privilege Escalation",
        "escalate privileges",
        "gain root privileges",
        "gain elevated privileges"
    ],

    "Information Disclosure": [
        "Information Disclosure",
        "information disclosure",
        "memory disclosure",
        "sensitive information disclosure",
        "information leak",
        "disclosure of local files"
    ],

    "Authentication Bypass": [
        "Authentication Bypass",
        "authentication bypass",
        "incorrect authentication",
        "bypass authentication",
        "credential verification bypass"
    ],

    "Authorization/Access Control Bypass": [
        "Authorization Bypass",
        "Access Control Bypass",
        "authorization bypass",
        "access control bypass",
        "bypass access restrictions",
        "bypass security restrictions"
    ],

    "Sandbox Escape": [
        "Sandbox Escape",
        "sandbox escape",
        "escape the sandbox"
    ],

    "SQL Injection": [
        "SQL Injection",
        "SQL injection"
    ],

    "Cross-Site Scripting": [
        "Cross-Site Scripting",
        "cross-site scripting",
        "XSS"
    ],

    "Server-Side Request Forgery": [
        "Server-Side Request Forgery",
        "server-side request forgery",
        "SSRF"
    ],

    "Path Traversal": [
        "Path Traversal",
        "path traversal",
        "directory traversal"
    ],

    "Command Injection": [
        "Command Injection",
        "command injection",
        "execution of arbitrary commands"
    ],

    "Request Smuggling": [
        "Request Smuggling",
        "request smuggling",
        "HTTP request smuggling"
    ],

    "Header Injection": [
        "Header Injection",
        "header injection"
    ],

    "Spoofing": [
        "Spoofing",
        "spoofing",
        "UI spoofing"
    ],

    "Cryptographic/Encryption Bypass": [
        "Cryptographic Bypass",
        "Encryption Bypass",
        "encryption bypass",
        "cryptographic bypass",
        "plaintext recovery"
    ]
}

### Función de clasificación

Por cada descripción:
1. **Limpieza**: sacamos los IDs de CVE (no aportan significado semántico) y normalizamos espacios.
2. **Clasificación por embeddings**: comparamos el embedding de la descripción contra el de cada categoría por similitud coseno; las que superan el umbral quedan seleccionadas.
3. **Respaldo por palabras clave**: además, buscamos literalmente los sinónimos del catálogo en el texto, por si el modelo de embeddings no detectó una categoría cuyo término aparece de forma explícita.

Aplicamos la función a todo el dataset y generamos una columna `tipo_vulnerabilidad` en formato texto (categorías separadas por coma) para inspección rápida.

In [9]:
# Cargar modelo de embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Pre-calcular embeddings usando la frase principal de cada categoría
nombres_categorias = list(categorias_mapeo.keys())
embeddings_categorias = model.encode(nombres_categorias)

def limpiar_y_clasificar(texto, umbral=0.28):
    if not texto or pd.isna(texto):
        return []

    # PASO 1: Limpieza drástica de patrones CVE y caracteres basura para eliminar ruido
    texto_limpio = re.sub(r'CVE-\d{4}-\d+', '', texto)
    texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()

    # PASO 2: Clasificación por Embeddings con umbral optimizado (0.28)
    embedding_texto = model.encode([texto_limpio])
    similitudes = cosine_similarity(embedding_texto, embeddings_categorias)[0]

    categorias_detectadas = set()
    for i, score in enumerate(similitudes):
        if score >= umbral:
            categorias_detectadas.add(nombres_categorias[i])

    # PASO 3: Respaldo por palabras clave para capturar términos explícitos omitidos
    texto_minusculas = texto.lower()
    for cat, sinonimos in categorias_mapeo.items():
        for sinonimo in sinonimos:
            if sinonimo.lower() in texto_minusculas:
                categorias_detectadas.add(cat)
                break

    return list(categorias_detectadas)

# Aplicar al DataFrame original
df["categorias_list"] = df["descripcion"].apply(lambda x: limpiar_y_clasificar(x))

# Para inspección rápida, creamos también una columna en formato texto (categorías separadas por coma).
# El CSV final para Looker Studio se genera en la siguiente sección con explode(), no acá.
df["tipo_vulnerabilidad"] = df["categorias_list"].apply(lambda x: ", ".join(x) if x else "Unclassified")

df.head(10)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25455.97it/s]


,fecha,package,cves,descripcion,dsa,url,categorias_list,tipo_vulnerabilidad
0,2026-09-19,chromium,CVE-2026-93372 CVE-2026-93373 CVE-2026-93374 C...,CVE-2026-93376 CVE-2026-93377 CVE-2026-93378 C...,DSA-6508-1,https://lists.debian.org/debian-security-annou...,"[Remote Code Execution, Information Disclosure...","Remote Code Execution, Information Disclosure,..."
1,2026-09-19,unbound,CVE-2026-14586 CVE-2026-32665 CVE-2026-40622 C...,CVE-2026-41637 CVE-2026-42955 CVE-2026-44621 C...,DSA-6507-1,https://lists.debian.org/debian-security-annou...,"[Remote Code Execution, Denial of Service]","Remote Code Execution, Denial of Service"
2,2026-09-17,chromium,CVE-2026-87429 CVE-2026-87430 CVE-2026-87431 C...,CVE-2026-87433 CVE-2026-87434 CVE-2026-87435 C...,DSA-6506-1,https://lists.debian.org/debian-security-annou...,"[Remote Code Execution, Information Disclosure...","Remote Code Execution, Information Disclosure,..."
3,2026-09-17,libapache2-mod-auth-openidc,CVE-2026-54789,It was discovered that incorrect state cookie ...,DSA-6504-1,https://lists.debian.org/debian-security-annou...,"[Authentication Bypass, Denial of Service]","Authentication Bypass, Denial of Service"
4,2026-09-17,bind9,CVE-2026-19033 CVE-2026-19662 CVE-2026-19666 C...,CVE-2026-19668 CVE-2026-75029 CVE-2026-76163 C...,DSA-6505-1,https://lists.debian.org/debian-security-annou...,[Denial of Service],Denial of Service
5,2026-09-16,thunderbird,CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 C...,CVE-2026-92009 CVE-2026-92010 CVE-2026-92011 C...,DSA-6503-1,https://lists.debian.org/debian-security-annou...,[Remote Code Execution],Remote Code Execution
6,2026-09-16,mkvtoolnix,CVE-2026-90783,A buffer overflow was found in the ODML parser...,DSA-6502-1,https://lists.debian.org/debian-security-annou...,[Remote Code Execution],Remote Code Execution
7,2026-09-16,firefox-esr,CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 C...,CVE-2026-92009 CVE-2026-92010 CVE-2026-92011 C...,DSA-6501-1,https://lists.debian.org/debian-security-annou...,"[Remote Code Execution, Privilege Escalation, ...","Remote Code Execution, Privilege Escalation, I..."
8,2026-09-16,tor,not yet available,Multiple security vulnerabilities were discove...,DSA-6500-1,https://lists.debian.org/debian-security-annou...,"[Authentication Bypass, Cryptographic/Encrypti...","Authentication Bypass, Cryptographic/Encryptio..."
9,2026-09-16,nginx,not yet available,The update for nginx released as DSA 6496-1 ca...,DSA-6496-2,https://lists.debian.org/debian-security-annou...,[],Unclassified


## 4. Preparación del dataset para Looker Studio

Como cada aviso puede tener varias categorías (`categorias_list`), eso está bien para un análisis exploratorio, pero para un dashboard de Looker Studio conviene tener **una fila por cada combinación (aviso, categoría)** en vez de una lista dentro de una celda — así se puede filtrar, contar y graficar por categoría directamente, sin lógica adicional del lado del BI. Usamos `explode()` para expandir la lista en filas, y completamos con "Unclassified" los avisos donde el clasificador no encontró ninguna categoría por encima del umbral, para no perderlos del conteo total.

In [10]:
# 1. Aseguramos que la columna sea una lista real de Python
# 2. Aplicamos explode para duplicar filas por cada elemento de la lista
df_looker_studio = df.explode("categorias_list")

# 3. Renombramos la columna para que quede clara en el tablero de control
df_looker_studio = df_looker_studio.rename(columns={"categorias_list": "vulnerabilidad_especifica"})

# 4. Rellenamos los vacíos que hayan quedado con 'Unclassified'
df_looker_studio["vulnerabilidad_especifica"] = df_looker_studio["vulnerabilidad_especifica"].fillna("Unclassified")

# 5. Exportamos el CSV limpio listo para conectar a Google Drive / Looker Studio
df_looker_studio.to_csv("debian_vulnerabilities_looker.csv", index=False)

print(f"Dataset expandido generado. Filas originales: {len(df)} | Filas expandidas para Looker: {len(df_looker_studio)}")


Dataset expandido generado. Filas originales: 154 | Filas expandidas para Looker: 393


Vista final del dataset listo para conectar a Looker Studio.

In [11]:
df_looker_studio.head(10)

,fecha,package,cves,descripcion,dsa,url,vulnerabilidad_especifica,tipo_vulnerabilidad
0,2026-09-19,chromium,CVE-2026-93372 CVE-2026-93373 CVE-2026-93374 C...,CVE-2026-93376 CVE-2026-93377 CVE-2026-93378 C...,DSA-6508-1,https://lists.debian.org/debian-security-annou...,Remote Code Execution,"Remote Code Execution, Information Disclosure,..."
0,2026-09-19,chromium,CVE-2026-93372 CVE-2026-93373 CVE-2026-93374 C...,CVE-2026-93376 CVE-2026-93377 CVE-2026-93378 C...,DSA-6508-1,https://lists.debian.org/debian-security-annou...,Information Disclosure,"Remote Code Execution, Information Disclosure,..."
0,2026-09-19,chromium,CVE-2026-93372 CVE-2026-93373 CVE-2026-93374 C...,CVE-2026-93376 CVE-2026-93377 CVE-2026-93378 C...,DSA-6508-1,https://lists.debian.org/debian-security-annou...,Denial of Service,"Remote Code Execution, Information Disclosure,..."
1,2026-09-19,unbound,CVE-2026-14586 CVE-2026-32665 CVE-2026-40622 C...,CVE-2026-41637 CVE-2026-42955 CVE-2026-44621 C...,DSA-6507-1,https://lists.debian.org/debian-security-annou...,Remote Code Execution,"Remote Code Execution, Denial of Service"
1,2026-09-19,unbound,CVE-2026-14586 CVE-2026-32665 CVE-2026-40622 C...,CVE-2026-41637 CVE-2026-42955 CVE-2026-44621 C...,DSA-6507-1,https://lists.debian.org/debian-security-annou...,Denial of Service,"Remote Code Execution, Denial of Service"
2,2026-09-17,chromium,CVE-2026-87429 CVE-2026-87430 CVE-2026-87431 C...,CVE-2026-87433 CVE-2026-87434 CVE-2026-87435 C...,DSA-6506-1,https://lists.debian.org/debian-security-annou...,Remote Code Execution,"Remote Code Execution, Information Disclosure,..."
2,2026-09-17,chromium,CVE-2026-87429 CVE-2026-87430 CVE-2026-87431 C...,CVE-2026-87433 CVE-2026-87434 CVE-2026-87435 C...,DSA-6506-1,https://lists.debian.org/debian-security-annou...,Information Disclosure,"Remote Code Execution, Information Disclosure,..."
2,2026-09-17,chromium,CVE-2026-87429 CVE-2026-87430 CVE-2026-87431 C...,CVE-2026-87433 CVE-2026-87434 CVE-2026-87435 C...,DSA-6506-1,https://lists.debian.org/debian-security-annou...,Denial of Service,"Remote Code Execution, Information Disclosure,..."
3,2026-09-17,libapache2-mod-auth-openidc,CVE-2026-54789,It was discovered that incorrect state cookie ...,DSA-6504-1,https://lists.debian.org/debian-security-annou...,Authentication Bypass,"Authentication Bypass, Denial of Service"
3,2026-09-17,libapache2-mod-auth-openidc,CVE-2026-54789,It was discovered that incorrect state cookie ...,DSA-6504-1,https://lists.debian.org/debian-security-annou...,Denial of Service,"Authentication Bypass, Denial of Service"


## 5. Pendiente para la entrega

Lo que todavía falta completar, según pide el enunciado:

- **Visualizaciones en Looker Studio**: crear al menos dos visualizaciones distintas a partir de `debian_vulnerabilities_looker.csv` (por ejemplo: cantidad de avisos por tipo de vulnerabilidad en el tiempo, y paquetes más afectados por categoría), y documentar el link al tablero, por qué se eligió cada visualización, y qué insight de seguridad aporta.
- **Justificación del manejo de datos para Looker**: documentar por qué se expandieron las filas con `explode()` (ver sección 4) como parte de esa justificación pedida por el enunciado.

## Nota sobre robustez

El scraping depende de que Debian no cambie el formato de texto de sus avisos (los patrones de fin de descripción, el formato de fecha, etc.). Si en el futuro se agregan nuevas distribuciones estables con un nombre distinto a "stable/oldstable/oldoldstable", los patrones de corte de la descripción (`patrones_fin`) van a necesitar actualizarse.

*Nota: el ajuste para que el scraper cruce el límite de año (sección 2) no se pudo probar contra el sitio en vivo en este entorno por falta de acceso a internet — conviene correrlo una vez y confirmar que sigue funcionando igual que antes.*